In [11]:
from app.backtest.BacktestEngine import BacktestEngine

engine = BacktestEngine()


Backtest engine initialized


In [12]:
import optuna

def objective_rsi(trial):
    rsi_period = trial.suggest_int('rsi_period', 7, 30)
    overbought = trial.suggest_int('overbought_threshold', 60, 85)
    oversold = trial.suggest_int('oversold_threshold', 15, 40)

    optimised_strategy = RSIStrategy(
        symbol="BTCUSDT",
        timeframe=Timeframe.ONE_MINUTE,
        rsi_period=rsi_period,
        overbought_threshold=overbought,
        oversold_threshold=oversold
    )

    try:
        optimised_result = engine.run(
            strategy=optimised_strategy,
            historical_data_file=historical_data_file,
            money_amount=initial_money,
            money_symbol=money_symbol,
            log_process=False
        )
        print(f"Trial finished. Result: {optimised_result}")
        return optimised_result.overall_profit_percent
    except Exception as e:
        print(f"Error during run: {e}")
        return -100.0

def objective_npos_rsi(trial):
    rsi_period = trial.suggest_int('rsi_period', 7, 30)
    overbought = trial.suggest_int('overbought_threshold', 60, 85)
    oversold = trial.suggest_int('oversold_threshold', 15, 40)
    npos = trial.suggest_int('max_positions', 10, 100)

    optimised_strategy = NPositionsRSIStrategy(
        symbol="BTCUSDT",
        timeframe=Timeframe.ONE_MINUTE,
        rsi_period=rsi_period,
        overbought_threshold=overbought,
        oversold_threshold=oversold,
        max_positions=npos
    )

    try:
        optimised_result = engine.run(
            strategy=optimised_strategy,
            historical_data_file=historical_data_file,
            money_amount=initial_money,
            money_symbol=money_symbol,
            log_process=False
        )
        print(f"Trial finished. Result: {optimised_result}")
        return optimised_result.overall_profit_percent
    except Exception as e:
        print(f"Error during run: {e}")
        return -100.0

In [13]:
from app.utils.timeframe import Timeframe
from app.strategies.implementation.RSIStrategy import RSIStrategy, NPositionsRSIStrategy

historical_data_file = r"backtest_data_btc_usdt_2020-01-01_00-00-00_2020-06-01_00-00-00_1m.csv"
initial_money = 1000
money_symbol = "BTCUSDT"

def run_rsi_strategy():
    optimised_strategy = RSIStrategy(
        symbol="BTCUSDT",
        timeframe=Timeframe.ONE_MINUTE,
        rsi_period=8,
        overbought_threshold=62,
        oversold_threshold=36
    )

    try:
        optimised_result = engine.run(
            strategy=optimised_strategy,
            historical_data_file=historical_data_file,
            money_amount=initial_money,
            money_symbol=money_symbol,
            log_process=False
        )
        print(f"Trial finished. Result: {optimised_result}")
        return optimised_result.overall_profit_percent
    except Exception as e:
        print(f"Error during run: {e}")
        return -100.0

def run_npos_rsi_strategy():
    optimised_strategy = NPositionsRSIStrategy(
        symbol="BTCUSDT",
        timeframe=Timeframe.ONE_MINUTE,
        rsi_period=8,
        overbought_threshold=62,
        oversold_threshold=36,
        max_positions=10
    )

    try:
        optimised_result = engine.run(
            strategy=optimised_strategy,
            historical_data_file=historical_data_file,
            money_amount=initial_money,
            money_symbol=money_symbol,
            log_process=False
        )
        print(f"Trial finished. Result: {optimised_result}")
        return optimised_result.overall_profit_percent
    except Exception as e:
        print(f"Error during run: {e}")
        return -100.0

In [14]:
run_rsi_strategy()


Running backtest for strategy 'RSIStrategy' w
Initial money: 1000 BTCUSDT
Backtest run completed
Trial finished. Result: Strategy name: RSIStrategy 
Initial amount: 1000.0 
Final amount: 2793.8138308604775 
Overall profit percent: 179.38138308604778 
Mean monthly profit percent: 0.0 
Max drawdown percent: 0.0 
Win rate: 70.57728119180634


np.float64(179.38138308604778)

In [15]:
study1 = optuna.create_study(direction='maximize')

study1.optimize(objective_rsi, n_trials=100)

print("Best parameters for objective_rsi:", study1.best_params)
print("Best profit:", study1.best_value)